# NewsGauge — Final production training pipeline for the Fakeddit fake-news BERT classifier

This notebook is the single source of truth for retraining the final model. It:

1. Writes the (audited, fixed) `src/data`, `src/calibration`, `src/training`, `src/evaluation` modules to disk from this notebook — this keeps the notebook and the committed package in sync, and is why you should run this notebook once and then `git add src/` (see the audit note in the repo README: previously these files existed only inside the old notebook and were never committed, which broke every documented training command).
2. Loads and splits the Fakeddit data **three ways** (train / validation / test) — the validation set is used for checkpoint selection and calibration; the test set is touched exactly once, at the end.
3. Fine-tunes BOTH `bert-base-uncased` and `distilbert-base-uncased` across 3 seeds each, so the final model choice is based on measured evidence, not assumption.
4. Selects the best checkpoint per run by validation F1, fits post-hoc temperature scaling (Guo et al., 2017) on the validation logits, and reports calibration (ECE, reliability diagrams) before/after.
5. Evaluates on the held-out test set exactly once per run, plots confusion matrices and training curves, and runs the existing heuristic error analysis.
6. Saves the selected final model with `save_pretrained()` / `tokenizer.save_pretrained()`, **reloads it from disk, and re-runs inference** to verify the saved artifact actually works before it goes anywhere near Hugging Face.

**Data**: this notebook does not download the Fakeddit TSVs itself (keeps it usable offline/behind restricted networks). Get `all_train.tsv`, `all_validate.tsv`, `all_test_public.tsv` from https://github.com/entitize/Fakeddit and point `CONFIG["data_dir"]` at the folder containing them. See `NEXT_STEPS.md` for exact instructions.

**Hardware**: a T4 GPU (e.g. Colab free tier or Kaggle) is enough — training both models across 3 seeds each on 5,000 balanced examples takes roughly 30–45 minutes total on a T4.

## 1. Setup

In [ ]:
# If running on a fresh Colab/Kaggle instance, uncomment:
# !pip install -q torch>=2.1 "transformers==4.41.2" datasets numpy pandas scikit-learn scipy captum matplotlib seaborn tabulate


In [ ]:
import os
import sys
import json
import random
import copy
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


## 2. Configuration — single source of truth for every hyperparameter used below

In [ ]:
CONFIG = {
    "data_dir": "data/fakeddit",          # folder containing all_train.tsv / all_validate.tsv / all_test_public.tsv
    "out_dir": "results/final_model",
    "n_per_class": 2500,                   # balanced sample size per class, before splitting
    "val_size": 0.1,                       # fraction of the full balanced set
    "test_size": 0.2,                      # fraction of the full balanced set (held out, evaluated once)
    "max_len": 64,
    "batch_size": 32,
    "epochs": 4,
    "lr": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "seeds": [42, 1, 7],                   # multi-seed evaluation, matches results/multiseed_summary.csv convention
    "candidate_models": ["bert-base-uncased", "distilbert-base-uncased"],
    "hf_repo_id": None,                    # set this to e.g. "your-username/fakeddit-bert-fake-news" before Section 8
}
os.makedirs(CONFIG["out_dir"], exist_ok=True)
CONFIG


## 3. Reproducibility

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(CONFIG["seeds"][0])


## 4. Write (and thereby restore) the package modules

Each cell below writes one file under `src/`. This is the same `%%writefile` convention the *original* notebook used — the difference is that this time you should actually commit the result (`git add src/data src/calibration src/training/engine.py src/training/train_transformer.py src/evaluation/error_analysis.py && git commit`) so the repository stays runnable outside this notebook too. Everything written here is identical to the corresponding file already provided alongside this notebook in `src/`.

**Important**: `%%writefile` does NOT create missing parent directories — it fails with `FileNotFoundError` if the target folder doesn't exist yet (this bites people running the notebook standalone, not from inside a full clone of the repo). The next cell creates every folder these `%%writefile` cells need, with `exist_ok=True` so it's always safe to run even if the folders already exist.

In [ ]:
# Must run before ANY %%writefile cell below, or those cells fail with
# FileNotFoundError if these folders don't already exist (e.g. fresh Colab
# runtime, or running this notebook without the rest of the repo cloned).
for _dir in ["src", "src/data", "src/calibration", "src/training", "src/evaluation", "app", "results"]:
    os.makedirs(_dir, exist_ok=True)
print("Directories ready:", ["src", "src/data", "src/calibration", "src/training", "src/evaluation", "app", "results"])


In [ ]:
%%writefile src/__init__.py


In [ ]:
%%writefile src/data/__init__.py


In [ ]:
%%writefile src/data/preprocess.py
"""Load, clean, balance and split the Fakeddit dataset.

Expects the three official TSVs (all_train.tsv, all_validate.tsv,
all_test_public.tsv) to already be present on disk -- see README.md /
NEXT_STEPS.md for where to get them. This module does not download data
itself so it stays usable in offline / restricted-network environments.

NOTE ON PROVENANCE: this file previously existed only inside
BERT_Fake_News_Full_Pipeline.ipynb (written via `%%writefile` when the
notebook was run in Colab) and was never committed to the repository, which
broke every script under src/training/ and two of the test modules. It is
restored here verbatim, plus one additive change: `split_three_way()` /
`prepare_dataset_with_val()`, used by the new production training path to
get a genuine held-out validation set (the original `split()` /
`prepare_dataset()` two-way API is left untouched so the existing test
suite in tests/test_preprocess.py keeps passing unmodified).
"""
from __future__ import annotations

import os
from dataclasses import dataclass

import pandas as pd
from sklearn.model_selection import train_test_split


@dataclass
class DatasetStats:
    original_total: int
    original_fake: int
    original_real: int
    balanced_total: int
    train_size: int
    test_size: int


@dataclass
class DatasetStatsWithVal:
    original_total: int
    original_fake: int
    original_real: int
    balanced_total: int
    train_size: int
    val_size: int
    test_size: int


def load_and_combine(data_dir: str, file_names=None) -> pd.DataFrame:
    """Load and concatenate the three Fakeddit TSVs.

    Note: Fakeddit ships its own train/validate/test split, and pooling all
    three before re-splitting (as this function's callers do) is a
    deliberate, documented choice for fast controlled comparison across
    models -- but it means results here are NOT directly comparable to
    numbers reported against Fakeddit's official test split in other papers.
    """
    file_names = file_names or ["all_train.tsv", "all_validate.tsv", "all_test_public.tsv"]
    paths = [os.path.join(data_dir, f) for f in file_names]
    missing = [p for p in paths if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(
            f"Missing Fakeddit files: {missing}. See NEXT_STEPS.md for download instructions."
        )
    dfs = [pd.read_csv(p, sep="\t", on_bad_lines="skip", low_memory=False) for p in paths]
    return pd.concat(dfs, ignore_index=True)


def _detect_title_column(df: pd.DataFrame) -> str:
    for col in df.columns:
        if "clean" in col.lower() and "title" in col.lower():
            return col
    for col in df.columns:
        if col.lower() == "title":
            return col
    raise ValueError("Could not find a title column in the dataset.")


def clean(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Keep only clean_title / label / id, drop nulls and near-empty titles."""
    title_col = _detect_title_column(raw_df)
    needed = [c for c in [title_col, "2_way_label", "id"] if c in raw_df.columns]
    df = raw_df[needed].copy()
    df.rename(columns={title_col: "clean_title", "2_way_label": "label"}, inplace=True)

    df.dropna(subset=["clean_title", "label"], inplace=True)
    df["clean_title"] = df["clean_title"].astype(str).str.strip()
    df = df[df["clean_title"].str.len() > 3].copy()
    df["label"] = df["label"].astype(int)
    df.reset_index(drop=True, inplace=True)
    return df


def balance(df: pd.DataFrame, n_per_class: int = 2500, seed: int = 42) -> pd.DataFrame:
    """Randomly sample n_per_class rows from each label to build a balanced set."""
    fake_df = df[df["label"] == 1].sample(n=n_per_class, random_state=seed)
    real_df = df[df["label"] == 0].sample(n=n_per_class, random_state=seed)
    return (
        pd.concat([fake_df, real_df], ignore_index=True)
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )


def split(df: pd.DataFrame, test_size: float = 0.2, seed: int = 42):
    """Stratified 80/20 (default) train/test split, keeps classes balanced in both.

    Kept exactly as-is (two-way only) for backward compatibility with the
    existing test suite and the baseline/scratch-model scripts. For the
    production BERT training path, prefer `split_three_way` below, which
    adds a genuine validation split instead of watching the test set during
    training.
    """
    train_df, test_df = train_test_split(
        df, test_size=test_size, random_state=seed, stratify=df["label"]
    )
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)


def split_three_way(df: pd.DataFrame, val_size: float = 0.1, test_size: float = 0.2, seed: int = 42):
    """Stratified train/validation/test split.

    val_size and test_size are both fractions of the *full* input `df`
    (not of the remaining train pool), so val_size=0.1, test_size=0.2 gives
    a 70/10/20 split. The validation split exists so checkpoint selection
    during training never looks at the test set, and the test set is
    evaluated exactly once, at the end.
    """
    if val_size + test_size >= 1.0:
        raise ValueError("val_size + test_size must be < 1.0")

    train_val_df, test_df = train_test_split(
        df, test_size=test_size, random_state=seed, stratify=df["label"]
    )
    # val_size was expressed as a fraction of the full df; convert it to a
    # fraction of what remains (train_val_df) so the final split proportions
    # match what the caller asked for.
    relative_val_size = val_size / (1.0 - test_size)
    train_df, val_df = train_test_split(
        train_val_df,
        test_size=relative_val_size,
        random_state=seed,
        stratify=train_val_df["label"],
    )
    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )


def prepare_dataset(
    data_dir: str,
    n_per_class: int = 2500,
    test_size: float = 0.2,
    seed: int = 42,
    train_fraction: float = 1.0,
):
    """Full pipeline: load -> clean -> balance -> split (two-way, unchanged).

    train_fraction: keep only this fraction of the *training* split (stratified),
    used for the data-scaling ablation (10% / 25% / 50% / 100%). Test set is
    always kept at full size so results across fractions are comparable.
    """
    raw_df = load_and_combine(data_dir)
    df = clean(raw_df)
    balanced_df = balance(df, n_per_class=n_per_class, seed=seed)
    train_df, test_df = split(balanced_df, test_size=test_size, seed=seed)

    if train_fraction < 1.0:
        train_df, _ = train_test_split(
            train_df,
            train_size=train_fraction,
            random_state=seed,
            stratify=train_df["label"],
        )
        train_df = train_df.reset_index(drop=True)

    stats = DatasetStats(
        original_total=len(df),
        original_fake=int((df["label"] == 1).sum()),
        original_real=int((df["label"] == 0).sum()),
        balanced_total=len(balanced_df),
        train_size=len(train_df),
        test_size=len(test_df),
    )
    return train_df, test_df, stats


def prepare_dataset_with_val(
    data_dir: str,
    n_per_class: int = 2500,
    val_size: float = 0.1,
    test_size: float = 0.2,
    seed: int = 42,
    train_fraction: float = 1.0,
):
    """Full pipeline with a genuine three-way split: load -> clean -> balance -> split.

    Use this (not `prepare_dataset`) for the final production model: the
    validation split lets you pick the best checkpoint and fit temperature
    scaling without ever touching the test set until the single final
    evaluation.
    """
    raw_df = load_and_combine(data_dir)
    df = clean(raw_df)
    balanced_df = balance(df, n_per_class=n_per_class, seed=seed)
    train_df, val_df, test_df = split_three_way(
        balanced_df, val_size=val_size, test_size=test_size, seed=seed
    )

    if train_fraction < 1.0:
        train_df, _ = train_test_split(
            train_df,
            train_size=train_fraction,
            random_state=seed,
            stratify=train_df["label"],
        )
        train_df = train_df.reset_index(drop=True)

    stats = DatasetStatsWithVal(
        original_total=len(df),
        original_fake=int((df["label"] == 1).sum()),
        original_real=int((df["label"] == 0).sum()),
        balanced_total=len(balanced_df),
        train_size=len(train_df),
        val_size=len(val_df),
        test_size=len(test_df),
    )
    return train_df, val_df, test_df, stats


In [ ]:
%%writefile src/data/dataset.py
"""PyTorch Dataset wrapping a tokenizer over the Fakeddit title/label pairs.

Restored from BERT_Fake_News_Full_Pipeline.ipynb (see preprocess.py header
for why this file was missing from the repository). Unchanged from the
notebook version.
"""
import pandas as pd
import torch
from torch.utils.data import Dataset


class FakedditDataset(Dataset):
    """Each item returns input_ids/attention_mask/token_type_ids/label tensors.

    Padding: right-padding to max_len with [PAD]=0.
    Truncation: right-truncation to max_len tokens.
    """

    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_len: int):
        self.texts = dataframe["clean_title"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx: int) -> dict:
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "token_type_ids": torch.zeros(self.max_len, dtype=torch.long),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


In [ ]:
%%writefile src/calibration/__init__.py


In [ ]:
%%writefile src/calibration/temperature_scaling.py
"""Post-hoc confidence calibration via temperature scaling.

Implements the method from Guo, Pleiss, Sun & Weinberger, "On Calibration of
Modern Neural Networks" (ICML 2017, https://arxiv.org/abs/1706.04599): a
single scalar temperature T > 0 is fit, by minimizing negative log-likelihood
on a held-out validation set, to rescale logits before softmax:

    p_calibrated(y=i | x) = softmax(z(x) / T)_i

T is fit ONLY on the validation split, never on the test split. Accuracy is
unchanged by temperature scaling (it's a monotonic rescaling, so argmax is
identical) -- only the reported confidence changes. Guo et al. found
temperature scaling matches or beats more complex calibration methods
(vector/matrix scaling, histogram binning) on this kind of classification
task while being far less prone to overfitting on small validation sets --
relevant here since Fakeddit's validation split in this project is only a
few hundred examples.

This module was added because the existing demo (app/demo_app.py) reports
raw softmax confidence directly to end users with no calibration check at
all -- a known failure mode for fine-tuned transformers on small training
sets (Guo et al. 2017, Section 1).
"""
from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


class TemperatureScaler(nn.Module):
    """Wraps a single learned scalar temperature applied to logits."""

    def __init__(self):
        super().__init__()
        # Start at T=1.0 (no-op) and optimize from there.
        self.temperature = nn.Parameter(torch.ones(1) * 1.0)

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return logits / self.temperature.clamp(min=1e-3)

    def fit(self, logits: torch.Tensor, labels: torch.Tensor, lr: float = 0.01, max_iter: int = 50) -> float:
        """Fit T by minimizing NLL on (logits, labels). Both must come from
        the validation set only. Returns the fitted temperature."""
        logits = logits.detach()
        labels = labels.detach()
        nll_criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.LBFGS([self.temperature], lr=lr, max_iter=max_iter)

        def closure():
            optimizer.zero_grad()
            loss = nll_criterion(self.forward(logits), labels)
            loss.backward()
            return loss

        optimizer.step(closure)
        return float(self.temperature.detach().clamp(min=1e-3).item())

    def calibrated_probs(self, logits: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            return F.softmax(self.forward(logits), dim=-1)


@dataclass
class ECEResult:
    ece: float
    bin_accuracies: list
    bin_confidences: list
    bin_counts: list
    bin_edges: list


def expected_calibration_error(probs: torch.Tensor, labels: torch.Tensor, n_bins: int = 15) -> ECEResult:
    """Expected Calibration Error (Naeini et al. 2015), as used in Guo et al. 2017.

    probs: (N, n_classes) softmax probabilities (raw OR calibrated -- call
    this twice, once per set, to compare before/after).
    labels: (N,) integer class labels.
    """
    confidences, predictions = probs.max(dim=-1)
    accuracies = predictions.eq(labels)

    bin_edges = torch.linspace(0, 1, n_bins + 1)
    ece = torch.zeros(1)
    bin_accs, bin_confs, bin_counts = [], [], []

    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        in_bin = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        count = int(in_bin.sum().item())
        bin_counts.append(count)
        if count > 0:
            acc = accuracies[in_bin].float().mean().item()
            conf = confidences[in_bin].mean().item()
            bin_accs.append(acc)
            bin_confs.append(conf)
            ece += (count / len(confidences)) * abs(acc - conf)
        else:
            bin_accs.append(0.0)
            bin_confs.append(0.0)

    return ECEResult(
        ece=float(ece.item()),
        bin_accuracies=bin_accs,
        bin_confidences=bin_confs,
        bin_counts=bin_counts,
        bin_edges=bin_edges.tolist(),
    )


def plot_reliability_diagram(result: ECEResult, title: str, ax=None):
    """Draws a standard reliability diagram (accuracy vs. confidence per bin)."""
    import matplotlib.pyplot as plt

    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))

    bin_centers = [(result.bin_edges[i] + result.bin_edges[i + 1]) / 2 for i in range(len(result.bin_accuracies))]
    ax.bar(bin_centers, result.bin_accuracies, width=1.0 / len(bin_centers), edgecolor="black", alpha=0.7, label="Accuracy")
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{title}\nECE = {result.ece:.4f}")
    ax.legend()
    return ax


In [ ]:
%%writefile src/evaluation/__init__.py


In [ ]:
%%writefile src/evaluation/error_analysis.py
"""Pull misclassified examples and bucket them by simple heuristic categories.

The categories are intentionally simple heuristics (length, punctuation,
lexical cues) meant as a *starting point* for manual inspection -- the
report should still eyeball the actual misclassified examples and describe
the error patterns in prose (see NEXT_STEPS.md Step 5).

CHANGE: `build_error_report` now takes an optional `seed` (default 42, so
existing callers keep identical behavior) instead of hardcoding
`random_state=42` regardless of the run's actual seed. Multi-seed runs now
produce genuinely different sampled error examples per seed.
"""
import pandas as pd

CLICKBAIT_CUES = {
    "shocking", "you won't believe", "wont believe", "unbelievable",
    "must see", "goes viral", "insane", "outrageous", "amazing",
}


def categorize_error(text: str) -> str:
    t = text.lower()
    n_words = len(t.split())
    if n_words <= 4:
        return "very_short_title"
    if "?" in t:
        return "question_headline"
    if any(cue in t for cue in CLICKBAIT_CUES):
        return "clickbait_language"
    if t.count("!") >= 1:
        return "exclamatory_headline"
    if n_words >= 20:
        return "long_title"
    return "other"


def build_error_report(
    test_df: pd.DataFrame,
    preds,
    labels,
    text_col: str = "clean_title",
    top_k: int = 10,
    seed: int = 42,
) -> pd.DataFrame:
    """Returns a DataFrame of misclassified rows with a heuristic category,
    sorted so the first `top_k` rows are good candidates to paste into the
    report as worked examples."""
    df = test_df.reset_index(drop=True).copy()
    df["true_label"] = labels
    df["pred_label"] = preds
    errors = df[df["true_label"] != df["pred_label"]].copy()
    errors["error_category"] = errors[text_col].apply(categorize_error)
    errors = errors.sample(frac=1, random_state=seed).reset_index(drop=True)
    return errors.head(top_k) if top_k else errors


def category_counts(errors: pd.DataFrame) -> pd.Series:
    return errors["error_category"].value_counts()


In [ ]:
%%writefile src/training/__init__.py


In [ ]:
%%writefile src/training/engine.py
"""Shared training / evaluation loops.

Works with both our from-scratch BertModel (which returns a dict with a
'logits' key) and HuggingFace models (which return an object with a .logits
attribute), via the `get_logits` helper.

`evaluate_with_logits` is new (additive): it's identical to `evaluate` but
also returns raw logits, which the production training path needs to fit
temperature scaling on the validation set and to run the single final test
evaluation. `train_one_epoch` and `evaluate` are unchanged so
train_baseline.py and train_scratch.py (and their tests) are unaffected.
"""
import torch
import torch.nn as nn


def get_logits(outputs):
    if isinstance(outputs, dict):
        return outputs["logits"]
    return outputs.logits


def train_one_epoch(model, loader, optimizer, scheduler, loss_fn, device, log_every=20):
    model.train()
    total_loss, n_correct, n_total = 0.0, 0, 0

    for step, batch in enumerate(loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        logits = get_logits(outputs)
        loss = loss_fn(logits, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        n_correct += (preds == labels).sum().item()
        n_total += labels.size(0)

        if log_every and (step + 1) % log_every == 0:
            print(
                f"  Step {step + 1:>4}/{len(loader)} "
                f"| Loss: {total_loss / (step + 1):.4f} "
                f"| Acc: {n_correct / n_total:.4f}"
            )

    return total_loss / len(loader), n_correct / n_total


@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, n_correct, n_total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        logits = get_logits(outputs)
        loss = loss_fn(logits, labels)
        preds = logits.argmax(dim=-1)

        total_loss += loss.item()
        n_correct += (preds == labels).sum().item()
        n_total += labels.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return total_loss / len(loader), n_correct / n_total, all_preds, all_labels


@torch.no_grad()
def evaluate_with_logits(model, loader, loss_fn, device):
    """Same as `evaluate`, but also returns the raw (uncalibrated) logits
    for every example, stacked into a single (N, n_classes) tensor on CPU.
    Needed for temperature-scaling fit and for computing calibrated
    confidence at inference time.
    """
    model.eval()
    total_loss, n_correct, n_total = 0.0, 0, 0
    all_preds, all_labels, all_logits = [], [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        logits = get_logits(outputs)
        loss = loss_fn(logits, labels)
        preds = logits.argmax(dim=-1)

        total_loss += loss.item()
        n_correct += (preds == labels).sum().item()
        n_total += labels.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_logits.append(logits.detach().cpu())

    stacked_logits = torch.cat(all_logits, dim=0)
    return total_loss / len(loader), n_correct / n_total, all_preds, all_labels, stacked_logits


In [ ]:
%%writefile app/streamlit_app.py
"""Streamlit demo for the Fakeddit fake-news BERT/DistilBERT classifier.

Replaces the previous Gradio demo (app/demo_app.py). Loads the final model
directly from the Hugging Face Hub (set MODEL_ID below after Section 8 of
notebooks/train_final_model.ipynb / the model-upload steps in NEXT_STEPS.md),
caches it once per process with st.cache_resource (not st.cache_data --
a loaded model is a resource, not serializable data; see Streamlit's own
caching docs: https://docs.streamlit.io/develop/concepts/architecture/caching),
and applies the temperature-scaled calibration saved alongside the model so
the confidence shown to users isn't raw, likely-overconfident softmax output.
"""
import json

import streamlit as st
import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# --- Configuration -----------------------------------------------------------

MODEL_ID = "your-username/fakeddit-bert-fake-news"  # <-- set this after uploading to the Hub
MAX_LEN = 64
EXAMPLES = [
    "Local city council approves new budget for public libraries",
    "You won't believe what this celebrity did - doctors are FURIOUS!!!",
    "Study finds moderate coffee consumption linked to lower mortality risk",
    "Scientists confirm the moon is actually a hologram, government admits",
]

# --- Model loading (cached once per process, not per interaction) ------------

@st.cache_resource(show_spinner="Loading model...")
def load_model_and_tokenizer(model_id: str):
    model = AutoModelForSequenceClassification.from_pretrained(model_id)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model.eval()
    try:
        cal_path = hf_hub_download(repo_id=model_id, filename="calibration.json")
        with open(cal_path) as f:
            calibration = json.load(f)
    except Exception:
        # Falls back to an uncalibrated temperature of 1.0 (= raw softmax) if
        # calibration.json isn't present in the Hub repo for some reason,
        # rather than crashing the whole app.
        calibration = {"temperature": 1.0, "id2label": {"0": "real", "1": "fake"}}
    return model, tokenizer, calibration


def predict(text: str, model, tokenizer, calibration: dict) -> dict:
    enc = tokenizer(text, max_length=MAX_LEN, padding="max_length", truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits
    temperature = float(calibration.get("temperature", 1.0))
    probs = F.softmax(logits / temperature, dim=-1)[0]
    pred_idx = int(probs.argmax())
    id2label = calibration.get("id2label", {"0": "real", "1": "fake"})
    return {
        "label": id2label[str(pred_idx)],
        "confidence": float(probs[pred_idx]),
        "prob_real": float(probs[0]),
        "prob_fake": float(probs[1]),
    }


# --- Page ----------------------------------------------------------------

st.set_page_config(page_title="Fakeddit Fake-News Classifier", page_icon="📰", layout="centered")
st.title("📰 Fakeddit Fake-News Classifier")
st.caption("BERT/DistilBERT fine-tuned on a balanced Fakeddit subset — 2-way (real/fake) headline classification.")

with st.expander("⚠️ Important limitations — read before trusting a prediction", expanded=False):
    st.markdown(
        """
- **This is not a general-purpose fact-checker.** It was trained only on Reddit post *titles* from the
  [Fakeddit](https://fakeddit.netlify.app/) dataset, using distant-supervision labels from the subreddit a
  post came from — not a human fact-check of each claim.
- It judges the **style and phrasing of a headline**, not the truth of the underlying claim. A real,
  accurately-reported headline that happens to sound sensational may still be flagged as "fake," and a
  fabricated claim written in a sober, neutral style may be flagged as "real."
- The reported confidence has been calibrated with temperature scaling (see the model card), but calibration
  reduces *average* overconfidence — it does not guarantee any single prediction is correct.
- It has not been evaluated on news from outside Reddit, on claims post-dating its training data, or on
  languages other than English.
        """
    )

model, tokenizer, calibration = load_model_and_tokenizer(MODEL_ID)

st.subheader("Try it")
example_choice = st.selectbox("Example headlines (or type your own below)", ["(custom)"] + EXAMPLES)
default_text = "" if example_choice == "(custom)" else example_choice
text = st.text_area("Headline text", value=default_text, height=100, max_chars=2000)

col1, col2 = st.columns([1, 3])
run = col1.button("Classify", type="primary", use_container_width=True)

if run:
    if not text.strip():
        st.warning("Enter some text first.")
    else:
        result = predict(text, model, tokenizer, calibration)
        label = result["label"]
        conf = result["confidence"]

        if label == "fake":
            st.error(f"**Predicted: FAKE** — {conf:.1%} confidence")
        else:
            st.success(f"**Predicted: REAL** — {conf:.1%} confidence")

        st.progress(result["prob_fake"], text=f"P(fake) = {result['prob_fake']:.1%}")

        st.markdown(
            f"This means the model's calibrated estimate is that a headline phrased this way resembles "
            f"**{label}** Reddit posts {conf:.1%} of the time, based on patterns in its training data — "
            f"not a verified fact-check of this specific claim."
        )

        with st.expander("Token-level explanation (Integrated Gradients)"):
            st.info(
                "Integrated Gradients attribution (see src/interpretability/integrated_gradients.py) is "
                "currently implemented for bert-base-uncased only, run offline as part of the research "
                "notebook — it is not wired into this live app because running it per-request would make "
                "the app too slow for Streamlit Community Cloud's shared CPU tier. See the model card's "
                "'Explainability' section for saved examples."
            )

st.divider()
with st.expander("Model & evaluation details"):
    st.markdown(
        f"""
- **Model**: `{MODEL_ID}`
- **Task**: 2-way classification (real vs. fake), Fakeddit dataset
- **Calibration**: temperature scaling, T = {calibration.get('temperature', 1.0):.3f}
  (fit on a held-out validation split; see the model card for before/after ECE)
- Full training methodology, dataset details, and measured results: see the model card on the Hugging Face
  Hub page for `{MODEL_ID}`, and `notebooks/train_final_model.ipynb` in the source repository.
        """
    )


In [ ]:
%%writefile app/requirements.txt
# Minimal dependency set for deploying app/streamlit_app.py to Streamlit
# Community Cloud. In the app's "Advanced settings", set
# "Requirements file path" to app/requirements.txt (root requirements.txt
# is for the research/training environment and pulls in torch's full CUDA
# build, captum, matplotlib, etc. -- unnecessary weight for a CPU-only demo
# that only loads a model and runs inference).
streamlit==1.37.0
torch==2.3.1
transformers==4.41.2
huggingface_hub==0.23.4


In [ ]:
sys.path.insert(0, os.path.abspath("."))
from src.data.preprocess import prepare_dataset_with_val
from src.data.dataset import FakedditDataset
from src.calibration.temperature_scaling import TemperatureScaler, expected_calibration_error, plot_reliability_diagram
from src.training.engine import evaluate_with_logits, train_one_epoch
from src.evaluation.error_analysis import build_error_report, category_counts
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report
import torch.nn.functional as F


def compute_metrics(labels, preds) -> dict:
    """Same implementation as src/evaluation/metrics.py::compute_metrics -- defined
    here directly (not imported) so it is available both to the smoke test below
    and to the full multi-seed run in Section 7, without a forward-reference."""
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision_weighted": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall_weighted": recall_score(labels, preds, average="weighted", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
        "confusion_matrix": confusion_matrix(labels, preds).tolist(),
        "classification_report": classification_report(labels, preds, target_names=["Real (0)", "Fake (1)"], zero_division=0),
    }


## 4b. Quick smoke test — run this before anything else

This cell does **not** need the real Fakeddit data or a GPU. It builds a tiny synthetic dataset in memory,
downloads a ~1MB randomly-initialized test model (`hf-internal-testing/tiny-random-bert` — the same tiny
checkpoint the `transformers` library's own test suite uses), and runs the *entire* pipeline end-to-end —
split → tokenize → one training step → validation eval → temperature-scaling fit → save → reload from disk →
predict — in under a minute. If this cell fails, something is wrong with the code/environment (missing
package, version mismatch, etc.) and you should fix that *before* spending 30-45 minutes on the real run in
Section 7. If this cell passes, the pipeline mechanics are sound and any later failure is almost certainly
about the real data (path, format) or GPU memory, not the code.

In [ ]:
import tempfile, shutil

def _write_fake_tsv(path, n_fake, n_real, start_id=0):
    rows = []
    for i in range(n_fake):
        rows.append({"clean_title": f"shocking fake headline number {start_id+i} with enough words", "2_way_label": 1, "id": f"f{start_id+i}"})
    for i in range(n_real):
        rows.append({"clean_title": f"real reported news story number {start_id+i} with enough words", "2_way_label": 0, "id": f"r{start_id+i}"})
    pd.DataFrame(rows).to_csv(path, sep="\t", index=False)

_smoke_dir = tempfile.mkdtemp()
_write_fake_tsv(os.path.join(_smoke_dir, "all_train.tsv"), 60, 60, 0)
_write_fake_tsv(os.path.join(_smoke_dir, "all_validate.tsv"), 20, 20, 1000)
_write_fake_tsv(os.path.join(_smoke_dir, "all_test_public.tsv"), 20, 20, 2000)

_str_tr, _str_va, _str_te, _str_stats = prepare_dataset_with_val(
    _smoke_dir, n_per_class=50, val_size=0.15, test_size=0.15, seed=0
)
assert len(_str_tr) + len(_str_va) + len(_str_te) == 100, "smoke test: split sizes do not add up"
assert set(_str_tr["id"]).isdisjoint(_str_va["id"]) and set(_str_tr["id"]).isdisjoint(_str_te["id"]) and set(_str_va["id"]).isdisjoint(_str_te["id"]), "smoke test: split leakage!"
print(f"Smoke data OK: train={len(_str_tr)} val={len(_str_va)} test={len(_str_te)}")


In [ ]:
from transformers import AutoModelForSequenceClassification as _SmokeModelCls, AutoTokenizer as _SmokeTokCls
from torch.utils.data import DataLoader as _SmokeDataLoader
from torch.optim import AdamW as _SmokeAdamW
import torch.nn as _smoke_nn

_SMOKE_MODEL_ID = "hf-internal-testing/tiny-random-bert"  # ~1MB, random weights, same interface as real BERT

_smoke_tok = _SmokeTokCls.from_pretrained(_SMOKE_MODEL_ID)
_smoke_model = _SmokeModelCls.from_pretrained(_SMOKE_MODEL_ID, num_labels=2)
_smoke_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_smoke_model.to(_smoke_device)

_smoke_train_loader = _SmokeDataLoader(FakedditDataset(_str_tr, _smoke_tok, max_len=16), batch_size=8, shuffle=True)
_smoke_val_loader = _SmokeDataLoader(FakedditDataset(_str_va, _smoke_tok, max_len=16), batch_size=8)
_smoke_test_loader = _SmokeDataLoader(FakedditDataset(_str_te, _smoke_tok, max_len=16), batch_size=8)

_smoke_optimizer = _SmokeAdamW(_smoke_model.parameters(), lr=5e-5)
_smoke_loss_fn = _smoke_nn.CrossEntropyLoss()

# One real training step through the ACTUAL engine.py functions (not a reimplementation).
_tr_loss, _tr_acc = train_one_epoch(_smoke_model, _smoke_train_loader, _smoke_optimizer, None, _smoke_loss_fn, _smoke_device, log_every=0)
print(f"Smoke train step OK: loss={_tr_loss:.4f} acc={_tr_acc:.4f}")

_va_loss, _va_acc, _va_preds, _va_labels, _va_logits = evaluate_with_logits(_smoke_model, _smoke_val_loader, _smoke_loss_fn, _smoke_device)
print(f"Smoke val eval OK: loss={_va_loss:.4f} acc={_va_acc:.4f}")

# Calibration, through the ACTUAL TemperatureScaler class.
_smoke_scaler = TemperatureScaler()
_smoke_T = _smoke_scaler.fit(_va_logits, torch.tensor(_va_labels))
assert _smoke_T > 0, "smoke test: fitted temperature should be positive"
print(f"Smoke calibration OK: T={_smoke_T:.3f}")

_te_loss, _te_acc, _te_preds, _te_labels, _te_logits = evaluate_with_logits(_smoke_model, _smoke_test_loader, _smoke_loss_fn, _smoke_device)
_smoke_metrics = compute_metrics(_te_labels, _te_preds)
_smoke_ece_before = expected_calibration_error(F.softmax(_te_logits, dim=-1), torch.tensor(_te_labels))
_smoke_ece_after = expected_calibration_error(_smoke_scaler.calibrated_probs(_te_logits), torch.tensor(_te_labels))
print(f"Smoke test eval OK: acc={_smoke_metrics['accuracy']:.4f} ECE {_smoke_ece_before.ece:.4f} -> {_smoke_ece_after.ece:.4f}")

# Save + reload + predict, through the ACTUAL save/reload code path the real run uses.
_smoke_ckpt = os.path.join(_smoke_dir, "smoke_checkpoint")
_smoke_model.save_pretrained(_smoke_ckpt)
_smoke_tok.save_pretrained(_smoke_ckpt)
with open(os.path.join(_smoke_ckpt, "calibration.json"), "w") as f:
    json.dump({"temperature": _smoke_T, "id2label": {"0": "real", "1": "fake"}}, f)

_smoke_reloaded_model = _SmokeModelCls.from_pretrained(_smoke_ckpt)
_smoke_reloaded_tok = _SmokeTokCls.from_pretrained(_smoke_ckpt)
_smoke_enc = _smoke_reloaded_tok("this is a smoke test headline", max_length=16, padding="max_length", truncation=True, return_tensors="pt")
with torch.no_grad():
    _smoke_out = _smoke_reloaded_model(input_ids=_smoke_enc["input_ids"], attention_mask=_smoke_enc["attention_mask"])
assert _smoke_out.logits.shape == (1, 2), f"smoke test: unexpected logits shape {_smoke_out.logits.shape}"
print("Smoke save/reload/inference OK:", F.softmax(_smoke_out.logits, dim=-1).tolist())

shutil.rmtree(_smoke_dir, ignore_errors=True)
print("\n*** SMOKE TEST PASSED - the pipeline mechanics work end-to-end. Safe to proceed to the real run. ***")


## 4c. Get the Fakeddit data automatically (no manual download)

This downloads the official v2.0 text/metadata dataset directly from the dataset authors' published Google
Drive folder (linked from https://github.com/entitize/Fakeddit, verified 2026-09-11:
https://drive.google.com/drive/folders/1jU7qgDqU1je9Y0PMKJ_f31yXRo5uWGFm) using `gdown`, which works for
publicly shared Drive folders with no login. It then searches whatever was downloaded for the train/validate/test
files by name pattern (the authors' folder structure/naming has changed before, so this doesn't assume an
exact layout) and copies them into `CONFIG["data_dir"]` with the canonical names `load_and_combine` expects.

**I could not run this cell myself before shipping it** (no network access to Google Drive from where this
notebook was written) — if it can't find the files automatically, it prints exactly what it *did* find so you
can see what's actually in the folder, rather than failing with a confusing error.

In [ ]:
# !pip install -q gdown   # uncomment if gdown isn't already available in your environment
import gdown
import glob
import shutil

FAKEDDIT_DRIVE_FOLDER = "https://drive.google.com/drive/folders/1jU7qgDqU1je9Y0PMKJ_f31yXRo5uWGFm"
_download_root = "fakeddit_raw_download"
os.makedirs(CONFIG["data_dir"], exist_ok=True)

# Skip re-downloading (this folder is large) if the canonical files are already in place.
_expected = ["all_train.tsv", "all_validate.tsv", "all_test_public.tsv"]
_already_have_all = all(os.path.exists(os.path.join(CONFIG["data_dir"], f)) for f in _expected)

if _already_have_all:
    print(f"Found all three files already in {CONFIG['data_dir']}/ - skipping download.")
else:
    print("Downloading the Fakeddit v2.0 text/metadata folder from Google Drive (this can take a while, it is a large folder)...")
    gdown.download_folder(url=FAKEDDIT_DRIVE_FOLDER, output=_download_root, quiet=False, use_cookies=False)

    # The authors' folder layout / naming has shifted over time, so search for the
    # files by pattern instead of assuming an exact path.
    all_tsvs = glob.glob(os.path.join(_download_root, "**", "*.tsv"), recursive=True)
    print(f"Found {len(all_tsvs)} .tsv files under {_download_root}/:")
    for p in all_tsvs:
        print(" ", p)

    def _find_one(patterns):
        for p in all_tsvs:
            name = os.path.basename(p).lower()
            if all(pat in name for pat in patterns):
                return p
        return None

    _train_src = _find_one(["train"]) 
    _val_src = _find_one(["valid"])   # matches "validate" or "valid"
    _test_src = _find_one(["test"])

    _found = {"all_train.tsv": _train_src, "all_validate.tsv": _val_src, "all_test_public.tsv": _test_src}
    _missing = [k for k, v in _found.items() if v is None]

    if _missing:
        raise FileNotFoundError(
            f"Could not auto-detect: {_missing} among the downloaded .tsv files listed above. "
            f"Open {_download_root}/ yourself, find the right train/validate/test files, and copy them to "
            f"{CONFIG['data_dir']}/all_train.tsv, all_validate.tsv, all_test_public.tsv manually -- "
            f"or edit the _find_one() patterns above to match the actual filenames printed."
        )

    for canonical_name, src_path in _found.items():
        shutil.copy(src_path, os.path.join(CONFIG["data_dir"], canonical_name))
        print(f"Copied {src_path} -> {CONFIG['data_dir']}/{canonical_name}")

    print("\nData download complete.")

for f in _expected:
    p = os.path.join(CONFIG["data_dir"], f)
    size_mb = os.path.getsize(p) / 1e6 if os.path.exists(p) else 0
    print(f"  {p}: {'OK, ' + format(size_mb, '.1f') + ' MB' if os.path.exists(p) else 'MISSING'}")


## 5. Load, clean, balance, and three-way split the data

In [ ]:
train_df, val_df, test_df, stats = prepare_dataset_with_val(
    CONFIG["data_dir"],
    n_per_class=CONFIG["n_per_class"],
    val_size=CONFIG["val_size"],
    test_size=CONFIG["test_size"],
    seed=CONFIG["seeds"][0],
)
print(stats)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Sanity check: no ID overlap between any pair of splits.
ids_train, ids_val, ids_test = set(train_df["id"]), set(val_df["id"]), set(test_df["id"])
assert ids_train.isdisjoint(ids_val)
assert ids_train.isdisjoint(ids_test)
assert ids_val.isdisjoint(ids_test)
print("No train/val/test ID overlap - confirmed.")

train_df["label"].value_counts()


## 6. Training + evaluation function (validation-based checkpoint selection, calibration, single test pass)

In [ ]:
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
import torch.nn as nn
# compute_metrics, F (torch.nn.functional) already defined/imported above (Section 4, used by the smoke test).


def train_and_evaluate(model_name: str, seed: int, train_df, val_df, test_df, config, verbose=True):
    """Full train -> validate -> select best checkpoint -> calibrate -> test-once pipeline for one (model, seed)."""
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_loader = DataLoader(FakedditDataset(train_df, tokenizer, config["max_len"]), batch_size=config["batch_size"], shuffle=True)
    val_loader = DataLoader(FakedditDataset(val_df, tokenizer, config["max_len"]), batch_size=config["batch_size"])
    test_loader = DataLoader(FakedditDataset(test_df, tokenizer, config["max_len"]), batch_size=config["batch_size"])

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

    no_decay = ["bias", "LayerNorm.weight"]
    param_groups = [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": config["weight_decay"]},
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
    ]
    optimizer = AdamW(param_groups, lr=config["lr"])
    total_steps = len(train_loader) * config["epochs"]
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(config["warmup_ratio"] * total_steps), num_training_steps=total_steps)
    loss_fn = nn.CrossEntropyLoss()

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_f1, best_state_dict, best_epoch = -1.0, None, -1

    for epoch in range(1, config["epochs"] + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, scheduler, loss_fn, device, log_every=0)
        va_loss, va_acc, va_preds, va_labels, _ = evaluate_with_logits(model, val_loader, loss_fn, device)
        va_metrics = compute_metrics(va_labels, va_preds)
        history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss); history["val_acc"].append(va_acc)
        if verbose:
            print(f"[{model_name} seed={seed}] epoch {epoch}/{config['epochs']} train_loss={tr_loss:.4f} val_acc={va_acc:.4f} val_f1={va_metrics['f1_weighted']:.4f}")
        if va_metrics["f1_weighted"] > best_val_f1:
            best_val_f1 = va_metrics["f1_weighted"]
            best_state_dict = copy.deepcopy(model.state_dict())
            best_epoch = epoch

    model.load_state_dict(best_state_dict)

    # Calibration is fit ONLY on validation logits from the best checkpoint.
    _, _, _, val_labels_final, val_logits_final = evaluate_with_logits(model, val_loader, loss_fn, device)
    scaler = TemperatureScaler()
    temperature = scaler.fit(val_logits_final, torch.tensor(val_labels_final))

    # Single final pass over the test set.
    te_loss, te_acc, preds, labels, test_logits = evaluate_with_logits(model, test_loader, loss_fn, device)
    metrics = compute_metrics(labels, preds)

    raw_probs = F.softmax(test_logits, dim=-1)
    calibrated_probs = scaler.calibrated_probs(test_logits)
    labels_t = torch.tensor(labels)
    ece_before = expected_calibration_error(raw_probs, labels_t)
    ece_after = expected_calibration_error(calibrated_probs, labels_t)

    return {
        "model_name": model_name,
        "seed": seed,
        "model": model,
        "tokenizer": tokenizer,
        "history": history,
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1,
        "temperature": temperature,
        "test_metrics": metrics,
        "test_preds": preds,
        "test_labels": labels,
        "ece_before": ece_before,
        "ece_after": ece_after,
    }


## 7. Run the multi-seed comparison across candidate models

This is the experiment that decides BERT-base vs. DistilBERT — based on the measured mean/std across `CONFIG["seeds"]`, not assumption. Each `train_and_evaluate` call trains one model from scratch (fine-tuning), so this cell runs `len(candidate_models) * len(seeds)` full training runs. Expect ~5-8 minutes per run on a T4.

In [ ]:
all_runs = []
for model_name in CONFIG["candidate_models"]:
    for seed in CONFIG["seeds"]:
        result = train_and_evaluate(model_name, seed, train_df, val_df, test_df, CONFIG)
        all_runs.append(result)
        print(
            f"DONE {model_name} seed={seed}: test_acc={result['test_metrics']['accuracy']:.4f} "
            f"test_f1={result['test_metrics']['f1_weighted']:.4f} "
            f"ECE {result['ece_before'].ece:.4f} -> {result['ece_after'].ece:.4f} (T={result['temperature']:.3f})"
        )


## 8. Aggregate results — mean ± std per model, statistical comparison

In [ ]:
summary_rows = []
for model_name in CONFIG["candidate_models"]:
    accs = [r["test_metrics"]["accuracy"] for r in all_runs if r["model_name"] == model_name]
    f1s = [r["test_metrics"]["f1_weighted"] for r in all_runs if r["model_name"] == model_name]
    eces = [r["ece_after"].ece for r in all_runs if r["model_name"] == model_name]
    summary_rows.append({
        "model": model_name,
        "n_runs": len(accs),
        "accuracy_mean": float(np.mean(accs)), "accuracy_std": float(np.std(accs)),
        "f1_mean": float(np.mean(f1s)), "f1_std": float(np.std(f1s)),
        "ece_after_calibration_mean": float(np.mean(eces)),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(CONFIG["out_dir"], "final_multiseed_summary.csv"), index=False)
summary_df


In [ ]:
from scipy import stats as sstats

bert_f1s = [r["test_metrics"]["f1_weighted"] for r in all_runs if r["model_name"] == "bert-base-uncased"]
distil_f1s = [r["test_metrics"]["f1_weighted"] for r in all_runs if r["model_name"] == "distilbert-base-uncased"]

if len(bert_f1s) > 1 and len(distil_f1s) > 1:
    t_stat, p_value = sstats.ttest_ind(bert_f1s, distil_f1s, equal_var=False)
    print(f"Welch's t-test on test F1 (bert-base vs distilbert), n={len(bert_f1s)} seeds each: t={t_stat:.3f}, p={p_value:.4f}")
    print("=> statistically indistinguishable at alpha=0.05" if p_value > 0.05 else "=> statistically significant difference at alpha=0.05")
else:
    print("Need >1 seed per model for a t-test; add more seeds to CONFIG['seeds'] if you want this.")


## 9. Select the final model

Rule: pick the model with the higher mean test F1 across seeds **unless** the difference is not statistically significant (p > 0.05) or is smaller than the observed std — in which case prefer DistilBERT for its ~40% smaller size and faster CPU inference on Streamlit Community Cloud, since accuracy is tied. This mirrors the existing repo's own documented, evidence-based reasoning in `results/multiseed_summary.csv` — verify the printed p-value above still supports it before trusting the cell below blindly on a different data sample.

In [ ]:
# Adjust this line manually based on the printed t-test result and summary_df above.
FINAL_MODEL_NAME = "distilbert-base-uncased"  # <-- change if bert-base-uncased wins with p < 0.05

# Pick the run with the median test F1 among the seeds for the chosen model,
# so the saved model is representative rather than a lucky outlier seed.
candidate_runs = [r for r in all_runs if r["model_name"] == FINAL_MODEL_NAME]
candidate_runs.sort(key=lambda r: r["test_metrics"]["f1_weighted"])
final_run = candidate_runs[len(candidate_runs) // 2]
print(f"Selected: {final_run['model_name']} seed={final_run['seed']} test_f1={final_run['test_metrics']['f1_weighted']:.4f}")


## 10. Plots — confusion matrix, training curves, reliability diagram

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(final_run["test_metrics"]["confusion_matrix"], annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred Real", "Pred Fake"], yticklabels=["Actual Real", "Actual Fake"], ax=ax)
ax.set_title(f"{final_run['model_name']} - Test Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], "confusion_matrix.png"), dpi=150)
plt.show()


In [ ]:
h = final_run["history"]
epochs_x = list(range(1, len(h["train_loss"]) + 1))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs_x, h["train_loss"], "b-o", label="Train"); ax1.plot(epochs_x, h["val_loss"], "r-s", label="Val")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Loss curves"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(epochs_x, h["train_acc"], "b-o", label="Train"); ax2.plot(epochs_x, h["val_acc"], "r-s", label="Val")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.set_title("Accuracy curves"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], "training_curves.png"), dpi=150)
plt.show()


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
plot_reliability_diagram(final_run["ece_before"], f"{final_run['model_name']} - before calibration", ax=ax1)
plot_reliability_diagram(final_run["ece_after"], f"{final_run['model_name']} - after calibration (T={final_run['temperature']:.3f})", ax=ax2)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["out_dir"], "reliability_diagram.png"), dpi=150)
plt.show()
print(f"Test ECE before calibration: {final_run['ece_before'].ece:.4f}")
print(f"Test ECE after calibration:  {final_run['ece_after'].ece:.4f}")


## 11. Error analysis

In [ ]:
errors = build_error_report(test_df, final_run["test_preds"], final_run["test_labels"], top_k=10, seed=final_run["seed"])
errors.to_csv(os.path.join(CONFIG["out_dir"], "error_examples.csv"), index=False)
print(category_counts(build_error_report(test_df, final_run["test_preds"], final_run["test_labels"], top_k=0, seed=final_run["seed"])))
errors[["clean_title", "true_label", "pred_label", "error_category"]]


## 12. Save the final model (Hugging Face-compatible)

`save_pretrained()` on both the model and tokenizer, plus the calibration temperature and label map, plus a metrics.json with everything measured above (no fabricated numbers — only what this notebook actually computed in Sections 7-11).

In [ ]:
FINAL_CKPT_DIR = os.path.join(CONFIG["out_dir"], "final_checkpoint")
os.makedirs(FINAL_CKPT_DIR, exist_ok=True)

final_run["model"].save_pretrained(FINAL_CKPT_DIR)
final_run["tokenizer"].save_pretrained(FINAL_CKPT_DIR)

calibration_payload = {
    "temperature": final_run["temperature"],
    "id2label": {"0": "real", "1": "fake"},
    "label2id": {"real": 0, "fake": 1},
    "test_ece_before_calibration": final_run["ece_before"].ece,
    "test_ece_after_calibration": final_run["ece_after"].ece,
}
with open(os.path.join(FINAL_CKPT_DIR, "calibration.json"), "w") as f:
    json.dump(calibration_payload, f, indent=2)

metrics_payload = dict(final_run["test_metrics"])
metrics_payload.update({
    "model_name": final_run["model_name"],
    "seed": final_run["seed"],
    "best_epoch": final_run["best_epoch"],
    "best_val_f1": final_run["best_val_f1"],
    "temperature": final_run["temperature"],
    "test_ece_before_calibration": final_run["ece_before"].ece,
    "test_ece_after_calibration": final_run["ece_after"].ece,
    "train_size": len(train_df), "val_size": len(val_df), "test_size": len(test_df),
    "config": CONFIG,
    "multiseed_summary": summary_df.to_dict(orient="records"),
})
with open(os.path.join(FINAL_CKPT_DIR, "training_metrics.json"), "w") as f:
    json.dump(metrics_payload, f, indent=2, default=str)

print("Saved to", FINAL_CKPT_DIR)
print(os.listdir(FINAL_CKPT_DIR))


## 13. Verify the saved artifact — reload from disk and run inference

This is the check the original repo's pipeline never had: prove the exact files just written to disk can be loaded back with the standard `AutoModelForSequenceClassification.from_pretrained` / `AutoTokenizer.from_pretrained` calls and produce sane, calibrated predictions — the same code path the Streamlit app and the Hugging Face Hub will use.

In [ ]:
from transformers import AutoModelForSequenceClassification as _AutoModel, AutoTokenizer as _AutoTok

reloaded_model = _AutoModel.from_pretrained(FINAL_CKPT_DIR)
reloaded_tokenizer = _AutoTok.from_pretrained(FINAL_CKPT_DIR)
with open(os.path.join(FINAL_CKPT_DIR, "calibration.json")) as f:
    reloaded_calibration = json.load(f)
reloaded_model.eval()

def predict(text, model, tokenizer, temperature, max_len=CONFIG["max_len"]):
    enc = tokenizer(text, max_length=max_len, padding="max_length", truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits
    calibrated = torch.softmax(logits / temperature, dim=-1)[0]
    pred_idx = int(calibrated.argmax())
    return {"label": ["real", "fake"][pred_idx], "confidence": float(calibrated[pred_idx])}

test_examples = [
    "Local city council approves new budget for public libraries",
    "You won't believe what scientists discovered - this changes everything!!!",
    "",
    "a" * 400,
]
for ex in test_examples:
    out = predict(ex, reloaded_model, reloaded_tokenizer, reloaded_calibration["temperature"])
    print(f"{out} <- {ex[:60]!r}{'...' if len(ex) > 60 else ''}")

print("\nReload + inference check passed: the saved checkpoint is loadable and produces valid predictions.")


## 14. Upload the model to Hugging Face Hub (automated)

Needs one thing from you: a Hugging Face token with **write** access, from https://huggingface.co/settings/tokens.
Everything else here is automated — repo creation, uploading the model/tokenizer/calibration files, and
generating the model card from the numbers this notebook actually measured (Sections 7-13), not a template
with placeholders.

In [ ]:
# !pip install -q huggingface_hub
from getpass import getpass
from huggingface_hub import HfApi, create_repo

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")  # if you saved it as a Colab secret named HF_TOKEN
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass("Paste a Hugging Face token with WRITE access (https://huggingface.co/settings/tokens): ")

HF_USERNAME = input("Your Hugging Face username: ").strip()
HF_MODEL_REPO = f"{HF_USERNAME}/fakeddit-bert-fake-news"
HF_SPACE_REPO = f"{HF_USERNAME}/newsgauge"

api = HfApi(token=HF_TOKEN)
print(f"Model repo:  {HF_MODEL_REPO}")
print(f"Space repo:  {HF_SPACE_REPO}")


In [ ]:
create_repo(HF_MODEL_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN)

api.upload_folder(
    folder_path=FINAL_CKPT_DIR,
    repo_id=HF_MODEL_REPO,
    repo_type="model",
    token=HF_TOKEN,
)
print(f"Uploaded model files from {FINAL_CKPT_DIR} to {HF_MODEL_REPO}")


### Generate the model card from the actual measured results (no placeholders)

In [ ]:
_bert_row = summary_df[summary_df["model"] == "bert-base-uncased"].iloc[0] if (summary_df["model"] == "bert-base-uncased").any() else None
_distil_row = summary_df[summary_df["model"] == "distilbert-base-uncased"].iloc[0] if (summary_df["model"] == "distilbert-base-uncased").any() else None

def _fmt(row, col):
    return f"{row[col]:.4f}" if row is not None else "N/A"

model_card = f"""---
language: en
license: mit
tags:
  - text-classification
  - fake-news-detection
  - {final_run["model_name"].split("-")[0]}
  - fakeddit
datasets:
  - fakeddit
metrics:
  - accuracy
  - f1
widget:
  - text: "Local city council approves new budget for public libraries"
  - text: "You won't believe what this celebrity did - doctors are FURIOUS!!!"
---

# Fakeddit Fake-News Classifier ({final_run["model_name"]})

Fine-tuned on a balanced subsample of [Fakeddit](https://fakeddit.netlify.app/) for binary (real/fake) headline
classification, with validation-based checkpoint selection and post-hoc temperature-scaled calibration.

**This model classifies headline style, not claim veracity. It is not a general-purpose fact-checker.**
See Limitations below.

## Training data

- Source: Fakeddit official train/validate/test TSVs, pooled and re-split (not directly comparable to papers
  using Fakeddit's own official test split).
- Balanced to {CONFIG["n_per_class"]} examples per class.
- Split: {len(train_df)} train / {len(val_df)} validation / {len(test_df)} test, stratified.

## Methodology

- Candidates fine-tuned across {len(CONFIG["seeds"])} seeds each: {", ".join(CONFIG["candidate_models"])}.
- Checkpoint selected by validation F1 (best epoch: {final_run["best_epoch"]}), test set evaluated exactly once.
- Post-hoc temperature scaling (Guo et al., 2017) fit on validation logits. Temperature T = {final_run["temperature"]:.4f}.
- Test ECE: {final_run["ece_before"].ece:.4f} (raw) -> {final_run["ece_after"].ece:.4f} (calibrated).

## Evaluation results (held-out test set, evaluated once)

| Metric | Value |
|---|---|
| Accuracy | {final_run["test_metrics"]["accuracy"]:.4f} |
| F1 (weighted) | {final_run["test_metrics"]["f1_weighted"]:.4f} |
| Precision (weighted) | {final_run["test_metrics"]["precision_weighted"]:.4f} |
| Recall (weighted) | {final_run["test_metrics"]["recall_weighted"]:.4f} |
| ECE (calibrated) | {final_run["ece_after"].ece:.4f} |

Multi-seed comparison (mean +/- std across {len(CONFIG["seeds"])} seeds):

| Model | Accuracy | F1 |
|---|---|---|
| bert-base-uncased | {_fmt(_bert_row, "accuracy_mean")} +/- {_fmt(_bert_row, "accuracy_std")} | {_fmt(_bert_row, "f1_mean")} +/- {_fmt(_bert_row, "f1_std")} |
| distilbert-base-uncased | {_fmt(_distil_row, "accuracy_mean")} +/- {_fmt(_distil_row, "accuracy_std")} | {_fmt(_distil_row, "f1_mean")} +/- {_fmt(_distil_row, "f1_std")} |

## Limitations

- Not a general-purpose fact-checker: labels come from Fakeddit's distant-supervision scheme (subreddit of
  origin), not per-claim human fact-checking.
- Trained only on short English Reddit post titles; untested on articles, other languages, or claims
  postdating training data collection.
- {len(train_df)+len(val_df)+len(test_df)} examples is a small, rebalanced subsample of Fakeddit's full corpus.
- Calibration reduces average overconfidence, not per-example correctness.

## How to use

```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch, torch.nn.functional as F

model = AutoModelForSequenceClassification.from_pretrained("{HF_MODEL_REPO}")
tokenizer = AutoTokenizer.from_pretrained("{HF_MODEL_REPO}")
text = "Local city council approves new budget for public libraries"
enc = tokenizer(text, max_length={CONFIG["max_len"]}, padding="max_length", truncation=True, return_tensors="pt")
with torch.no_grad():
    logits = model(**enc).logits
probs = F.softmax(logits / {final_run["temperature"]:.4f}, dim=-1)  # calibrated confidence
print(probs)
```

## Citation

Nakamura, K., Levy, S., & Wang, W. Y. (2020). r/Fakeddit: A New Multimodal Benchmark Dataset for Fine-grained
Fake News Detection. *Proceedings of LREC 2020.*
"""

api.upload_file(
    path_or_fileobj=model_card.encode("utf-8"),
    path_in_repo="README.md",   # Hugging Face renders README.md as the model card
    repo_id=HF_MODEL_REPO,
    repo_type="model",
    token=HF_TOKEN,
)
print(f"Model card uploaded. View at: https://huggingface.co/{HF_MODEL_REPO}")


## 15. Deploy the Streamlit frontend — as a Hugging Face Space, via Docker (no browser click-through needed)

**Update**: Hugging Face deprecated the native `space_sdk="streamlit"` option in April 2025 -- Spaces now only
accept `"gradio"`, `"docker"`, or `"static"`. Streamlit apps are deployed via the **Docker** SDK instead: a
small `Dockerfile` that installs Streamlit and runs it. This still runs on the free `cpu-basic` tier — no paid
plan needed. Streamlit Community Cloud (share.streamlit.io) is a separate option that still supports Streamlit
natively, but it requires a manual GitHub-OAuth "connect repo" click in a browser with no API, so it can't be
fully automated from here; `NEXT_STEPS.md` has those steps if you want that route instead.

In [ ]:
streamlit_template_path = "app/streamlit_app.py"
with open(streamlit_template_path) as f:
    streamlit_app_code = f.read()

# app/streamlit_app.py ships with a placeholder MODEL_ID -- swap in the real one.
placeholder = 'MODEL_ID = "your-username/fakeddit-bert-fake-news"  # <-- set this after uploading to the Hub'
replacement = f'MODEL_ID = "{HF_MODEL_REPO}"'
assert placeholder in streamlit_app_code, "placeholder MODEL_ID line not found -- app/streamlit_app.py may have changed"
streamlit_app_code = streamlit_app_code.replace(placeholder, replacement)

with open("app/requirements.txt") as f:
    space_requirements = f.read()

space_dockerfile = (
    "FROM python:3.10-slim\n\n"
    "WORKDIR /app\n\n"
    "COPY requirements.txt .\n"
    "RUN pip install --no-cache-dir -r requirements.txt\n\n"
    "COPY streamlit_app.py .\n\n"
    "EXPOSE 8501\n\n"
    "HEALTHCHECK CMD curl --fail http://localhost:8501/_stcore/health || exit 1\n\n"
    "ENTRYPOINT [\"streamlit\", \"run\", \"streamlit_app.py\", \"--server.port=8501\", \"--server.address=0.0.0.0\"]\n"
)

space_readme = (
    "---\n"
    "title: NewsGauge\n"
    "emoji: \U0001F4F0\n"
    "colorFrom: blue\n"
    "colorTo: red\n"
    "sdk: docker\n"
    "app_port: 8501\n"
    "pinned: false\n"
    "---\n\n"
    f"Live demo of {HF_MODEL_REPO} -- see that model's card for full methodology, evaluation results, and limitations.\n"
)

try:
    from huggingface_hub.utils import HfHubHTTPError
except ImportError:
    HfHubHTTPError = Exception

try:
    create_repo(HF_SPACE_REPO, repo_type="space", space_sdk="docker", exist_ok=True, token=HF_TOKEN)

    for filename, content in [
        ("streamlit_app.py", streamlit_app_code),
        ("Dockerfile", space_dockerfile),
        ("README.md", space_readme),
        ("requirements.txt", space_requirements),
    ]:
        api.upload_file(
            path_or_fileobj=content.encode("utf-8"),
            path_in_repo=filename,
            repo_id=HF_SPACE_REPO,
            repo_type="space",
            token=HF_TOKEN,
        )

    print("Space deployed (Docker SDK). It will build for a few minutes (Docker builds are slower than the old")
    print("native Streamlit runtime), then be live at:")
    print(f"  https://huggingface.co/spaces/{HF_SPACE_REPO}")
except HfHubHTTPError as e:
    if "402" in str(e) or "Payment Required" in str(e):
        print("Hugging Face returned 402 Payment Required: creating a new Gradio/Docker Space now requires")
        print("a paid HF plan (Static Spaces remain free, but can't run this Python app). Two free-tier-")
        print("compatible options instead:")
        print("  1) Subscribe to HF PRO (~$9/mo) at https://huggingface.co/pro, then re-run this cell.")
        print("  2) Deploy to Streamlit Community Cloud instead (free) -- see NEXT_STEPS.md Section 8.")
        print(f"     app/streamlit_app.py already has MODEL_ID set to \"{HF_MODEL_REPO}\".")
    else:
        raise


## 16. Done

- Model: `https://huggingface.co/{HF_MODEL_REPO}` (filled in above once Section 14 runs)
- Live Streamlit demo: `https://huggingface.co/spaces/{HF_SPACE_REPO}` (filled in above once Section 15 runs,
  give it a minute or two to build the first time)
- Everything above was generated from what this notebook actually measured — if you re-run it with different
  data/seeds, re-run Sections 14-15 too so the deployed model/card/demo match your latest numbers.